# Natural Language Understanding

In this section we go through topics related to text understanding. We cover such topics like:
    
- Similarity measures
- Word Vectors
- Vector Space Model
- Type of vectorizers
- Build a vectorizer with Tensorflow.

## Similarity measures

Word does have different meanings. This makes the comparison and analysis a bit more complex.

In [1]:
from textblob import Word

w = Word("developer")

for synset, definition in zip(w.get_synsets(), w.define()):
    print(synset, definition)

Synset('developer.n.01') someone who develops real estate (especially someone who prepares a site for residential or commercial use)
Synset('developer.n.02') photographic equipment consisting of a chemical solution for developing film


## Similarity measures

There are plenty of methods to measure the similarity of strings. Two most popular Python libraries examples for such measure are shown. We compare two strings: trains and training. The SequenceMatcher class allow us to use the Gestalt pattern matching algorithm:

In [4]:
from difflib import SequenceMatcher
a = "training"
b = "trains"
print(len(a))
print(len(b))
ratio = SequenceMatcher(None, a, b).ratio()
print(ratio)

8
6
0.7142857142857143


The distance is a normalized value between 0 and 1, where 1 means identical.

A different approach is shown below. We use the Jellyfish library. There are a few methods that we can use here. One of it is the Levenshtein distance. Below the distance and normalize distance values are calculated.

In [3]:
import jellyfish
distance = jellyfish.levenshtein_distance(a,b)
print(distance)

normalized_distance = distance/max(len(a),len(b))
print(1.0-normalized_distance)

3
0.625


Some words can be more similar to each other than other. We can build a similarity matrix to check it where 1 mean equal and 0 totally different.

In [7]:
import spacy

nlp = spacy.load("en_core_web_sm")

tokens = nlp(u'king queen horse cat desk lamp')

for first_token in tokens:
    for second_token in tokens:
        print(first_token.text, second_token.text, first_token.similarity(second_token))

king king 1.0
king queen 0.5523892045021057
king horse 0.6490848660469055
king cat 0.635511577129364
king desk 0.621141254901886
king lamp 0.3553648293018341
queen king 0.5523892045021057
queen queen 1.0
queen horse 0.6413618326187134
queen cat 0.6422978043556213
queen desk 0.4828205704689026
queen lamp 0.2853463590145111
horse king 0.6490848660469055
horse queen 0.6413618326187134
horse horse 1.0
horse cat 0.7530450820922852
horse desk 0.7113963961601257
horse lamp 0.28195956349372864
cat king 0.635511577129364
cat queen 0.6422978043556213
cat horse 0.7530450820922852
cat cat 1.0
cat desk 0.7171130180358887
cat lamp 0.26585325598716736
desk king 0.621141254901886
desk queen 0.4828205704689026
desk horse 0.7113963961601257
desk cat 0.7171130180358887
desk desk 1.0
desk lamp 0.3512752056121826
lamp king 0.3553648293018341
lamp queen 0.2853463590145111
lamp horse 0.28195956349372864
lamp cat 0.26585325598716736
lamp desk 0.3512752056121826
lamp lamp 1.0


/tmp/ipykernel_38416/4273872675.py:9: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Token.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  print(first_token.text, second_token.text, first_token.similarity(second_token))


We can also compare sentences:

In [12]:
doc1 = nlp(u"Warsaw is the largest city in Poland.")
doc2 = nlp(u"Crossaint is baked in France.")
doc3 = nlp(u"An emu is a large bird.")

for doc in [doc1, doc2, doc3]:
    for other_doc in [doc1, doc2, doc3]:
        print(doc.similarity(other_doc))

1.0
0.7503145336623088
0.6300677816449679
0.7503145336623088
1.0
0.5422713398307065
0.6300677816449679
0.5422713398307065
1.0


/tmp/ipykernel_38416/2306317000.py:7: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  print(doc.similarity(other_doc))


The similarity matrix looks like following:

|       | doc1 | doc2 | doc3 |
|-------|------|------|------|
| **doc1** | 1.0  | 0.72 | 0.65 |
| **doc 2** | 0.72 | 1.0  | 0.40 |
| **doc 3** | 0.65 | 0.40 | 1.0  |

## Word Vectors

SpaCy does have already a set of words that are vectorized.

Let's take a look at the vectors that are available in spaCy using the previous example:

In [8]:
nlp = spacy.load("en_core_web_sm")

tokens = nlp(u'king queen horse cat desk lamp')

for token in tokens:
    print(str(token)+" "+str(token.vector))

king [-0.33849952 -0.03223161  0.2951869   0.59571135  0.22842333 -0.24313462
  1.2632756   0.40882614  0.2996106   0.38803434  0.4805932  -0.52212375
 -0.56689256  0.21759133 -0.47461712  0.15028867 -0.06043254  0.17063658
  1.1439681  -1.3166867  -0.23129597  1.1934106  -0.8484381   0.52144444
  1.3578582   0.23143867 -0.17714147  0.58077836 -0.7838124   0.28310835
 -0.5557947  -0.5037915  -0.45771408 -0.22190154 -1.159611   -0.3153508
 -0.6977454   0.18609406 -0.46583915  1.1962719  -0.15789653  1.0093553
 -0.95039546  0.99920166 -0.09383069  0.6180186  -0.8498031   0.22529966
 -0.975485   -0.57056135 -0.4995728   0.25432923 -0.18943356 -0.6571157
 -0.21509902 -0.8799945   0.48541677  0.61795866  0.41183537  0.3287104
 -0.8590182  -0.10632116  0.34067023 -0.912362   -1.270268   -0.16583735
 -0.11312183  1.4501237  -0.08616459 -1.0183003   0.36685124  0.7115195
  0.81917906 -0.24078473  0.9165527  -0.6053555  -0.6809914  -0.6792679
  0.9947966   0.5872688  -0.33952343 -0.45329452 -0.

It looks that the vectors are quite long. It's easy to check the exact size of a vector:

In [10]:
len(tokens[1].vector)

96

You can play around and check the vector values for some other sentences. Let's take a look at sentence vectors of one of our previous examples:

In [13]:
len(doc1.vector)

96

A nice example of word vectorization done by some researchers at Warsaw University: [Word2Vec](https://lamyiowce.github.io/word2viz/).

## Negative sampling

It is a simpler implementation of word2vec. It is faster as it takes only a few terms in each iteration for training insted of the whole dataset as in previous example. This is why it's called negative sampling.

First of all, we define helper methods that are used later.

In [14]:
def zeros(*dims):
    return np.zeros(shape=tuple(dims), dtype=np.float32)

def ones(*dims):
    return np.ones(shape=tuple(dims), dtype=np.float32)

def rand(*dims):
    return np.random.rand(*dims).astype(np.float32)

def randn(*dims):
    return np.random.randn(*dims).astype(np.float32)

def sigmoid(batch, stochastic=False):
    return  1.0 / (1.0 + np.exp(-batch))

def as_matrix(vector):
    return np.reshape(vector, (-1, 1))

We need to load the data again.

In [16]:
import nltk
import numpy as np
import pandas as pd
from collections import namedtuple

nltk.download('all')

from nltk.book import *

texts()

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/tomek-ste/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/tomek-ste/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /home/tomek-ste/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_ta

[nltk_data]    |   Unzipping corpora/mte_teip5.zip.
[nltk_data]    | Downloading package mwa_ppdb to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Unzipping misc/mwa_ppdb.zip.
[nltk_data]    | Downloading package names to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Package names is already up-to-date!
[nltk_data]    | Downloading package nombank.1.0 to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    | Downloading package nonbreaking_prefixes to
[nltk_data]    |     /home/tomek-ste/nltk_data...
[nltk_data]    |   Unzipping corpora/nonbreaking_prefixes.zip.
[nltk_data]    | Downloading package nps_chat to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Package nps_chat is already up-to-date!
[nltk_data]    | Downloading package omw to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    | Downloading package omw-1.4 to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    | 

[nltk_data]    | Downloading package verbnet to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Unzipping corpora/verbnet.zip.
[nltk_data]    | Downloading package verbnet3 to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Unzipping corpora/verbnet3.zip.
[nltk_data]    | Downloading package webtext to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Package webtext is already up-to-date!
[nltk_data]    | Downloading package wmt15_eval to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Unzipping models/wmt15_eval.zip.
[nltk_data]    | Downloading package word2vec_sample to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Unzipping models/word2vec_sample.zip.
[nltk_data]    | Downloading package wordnet to /home/tomek-
[nltk_data]    |     ste/nltk_data...
[nltk_data]    |   Package wordnet is already up-to-date!
[nltk_data]    | Downloading package wordnet2021 to /home/tomek-
[nl

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


Three variables are important for the training: ``train_dict``, ``train_tokens`` and ``train_set``. The first one contain all unique words used in the corpus. The second is a list of indices of words in the dictionary that correspond to each word used in the raw text. 

In [17]:
#raw_set = nltk.corpus.treebank_raw.raw()[0:50000].replace('.START',' ').replace("\n","").replace("."," ").replace(","," ")
#tokens = [token for token in nltk.word_tokenize(raw_set) if token.isalpha()]
tokens = text6.tokens
train_dict = pd.Series(tokens).unique().tolist()
train_tokens = np.array([train_dict.index(token) for token in tokens])

In [29]:
train_tokens.shape

(16967,)

The last variable consist of a list of two numbers. The current word index and the word index that is before the word and after the word. Depending on the window size we use also other words that are in the neighbourhood. In this example the window size is set to 2. It means we take two words before and two words after the given word and build the relation in the training data set.

In [20]:
train_set = []
for i in range(2,len(tokens)-2):
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i-1])])
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i-2])])
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i+1])])
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i+2])])

train_set = np.random.permutation(np.array(train_set))

The next step is to set the training configuration. We set the the negative samples size to 10 and the vector size to 100. Learning rate and rate decay are set to 0.1 and 0.995. The training loops are set to 8000000. Logs are displayed each 10000 epoches.

In [28]:
Config = namedtuple("Config", ["dict_size", "vect_size", "neg_samples", "updates", "learning_rate",
                               "learning_rate_decay", "decay_period", "log_period"])
conf = Config(
    dict_size=len(train_dict),
    vect_size=100,
    neg_samples=10,
    updates=8000000,
    learning_rate=0.1,
    learning_rate_decay=0.995,
    decay_period=10000,
    log_period=10000)

We loop over ``updates`` and get the word and context from the train set. We calculate the negative context and calculate the word, context and negative sample vectors. The negative context is chosen randomly. In the next step we calcualte the cost and corresponding to it gradients.

In [30]:
def neg_sample(conf, train_set, train_tokens):
    Vp = randn(conf.dict_size, conf.vect_size)
    Vo = randn(conf.dict_size, conf.vect_size)

    J = 0.0
    learning_rate = conf.learning_rate
    for i in range(conf.updates):
        idx = i % len(train_set)

        word = train_set[idx, 0]
        context = train_set[idx, 1]

        neg_context = np.random.randint(0, len(train_tokens), conf.neg_samples)
        neg_context = train_tokens[neg_context]

        word_vect = Vp[word, :]  # word vector
        context_vect = Vo[context, :];  # context wector
        negative_vects = Vo[neg_context, :]  # sampled negative vectors

        # Cost and gradient calculation starts here
        score_pos = word_vect @ context_vect.T
        score_neg = word_vect @ negative_vects.T

        J -= np.log(sigmoid(score_pos)) + np.sum(np.log(sigmoid(-score_neg)))
        if (i + 1) % conf.log_period == 0:
            print('Update {0}\tcost: {1:>2.2f}'.format(i + 1, J / conf.log_period))
            final_cost = J / conf.log_period
            J = 0.0

        pos_g = 1.0 - sigmoid(score_pos)
        neg_g = sigmoid(score_neg)

        word_grad = -pos_g * context_vect + np.sum(as_matrix(neg_g) * negative_vects, axis=0)
        context_grad = -pos_g * word_vect
        neg_context_grad = as_matrix(neg_g) * as_matrix(word_vect).T

        Vp[word, :] -= learning_rate * word_grad
        Vo[context, :] -= learning_rate * context_grad
        Vo[neg_context, :] -= learning_rate * neg_context_grad

        if i % conf.decay_period == 0:
            learning_rate = learning_rate * conf.learning_rate_decay

    return Vp, Vo, final_cost

Next do the training:

In [31]:
Vp, Vo, J = neg_sample(conf, train_set, train_tokens)

Update 10000	cost: 18.76
Update 20000	cost: 10.74
Update 30000	cost: 8.59
Update 40000	cost: 7.36
Update 50000	cost: 6.57
Update 60000	cost: 6.11
Update 70000	cost: 5.62
Update 80000	cost: 4.85
Update 90000	cost: 4.65
Update 100000	cost: 4.45
Update 110000	cost: 4.34
Update 120000	cost: 4.20
Update 130000	cost: 4.14
Update 140000	cost: 4.00
Update 150000	cost: 3.81
Update 160000	cost: 3.77
Update 170000	cost: 3.62
Update 180000	cost: 3.65
Update 190000	cost: 3.57
Update 200000	cost: 3.57
Update 210000	cost: 3.47
Update 220000	cost: 3.39
Update 230000	cost: 3.37
Update 240000	cost: 3.33
Update 250000	cost: 3.32
Update 260000	cost: 3.24
Update 270000	cost: 3.26
Update 280000	cost: 3.21
Update 290000	cost: 3.14
Update 300000	cost: 3.15
Update 310000	cost: 3.10
Update 320000	cost: 3.12
Update 330000	cost: 3.05
Update 340000	cost: 3.09
Update 350000	cost: 3.02
Update 360000	cost: 2.99
Update 370000	cost: 2.98
Update 380000	cost: 2.96
Update 390000	cost: 2.92
Update 400000	cost: 2.91
Update 

Update 3210000	cost: 2.04
Update 3220000	cost: 2.05
Update 3230000	cost: 2.05
Update 3240000	cost: 2.04
Update 3250000	cost: 2.05
Update 3260000	cost: 2.06
Update 3270000	cost: 2.03
Update 3280000	cost: 2.05
Update 3290000	cost: 2.03
Update 3300000	cost: 2.04
Update 3310000	cost: 2.03
Update 3320000	cost: 2.07
Update 3330000	cost: 2.05
Update 3340000	cost: 2.03
Update 3350000	cost: 2.06
Update 3360000	cost: 2.04
Update 3370000	cost: 2.04
Update 3380000	cost: 2.05
Update 3390000	cost: 2.06
Update 3400000	cost: 2.03
Update 3410000	cost: 2.04
Update 3420000	cost: 2.04
Update 3430000	cost: 2.05
Update 3440000	cost: 2.04
Update 3450000	cost: 2.02
Update 3460000	cost: 2.05
Update 3470000	cost: 2.02
Update 3480000	cost: 2.00
Update 3490000	cost: 2.05
Update 3500000	cost: 2.05
Update 3510000	cost: 2.04
Update 3520000	cost: 2.03
Update 3530000	cost: 2.05
Update 3540000	cost: 2.02
Update 3550000	cost: 2.04
Update 3560000	cost: 2.03
Update 3570000	cost: 2.05
Update 3580000	cost: 2.03
Update 35900

Update 6370000	cost: 1.94
Update 6380000	cost: 1.95
Update 6390000	cost: 1.94
Update 6400000	cost: 1.94
Update 6410000	cost: 1.94
Update 6420000	cost: 1.95
Update 6430000	cost: 1.93
Update 6440000	cost: 1.96
Update 6450000	cost: 1.95
Update 6460000	cost: 1.93
Update 6470000	cost: 1.93
Update 6480000	cost: 1.95
Update 6490000	cost: 1.94
Update 6500000	cost: 1.94
Update 6510000	cost: 1.96
Update 6520000	cost: 1.94
Update 6530000	cost: 1.93
Update 6540000	cost: 1.93
Update 6550000	cost: 1.94
Update 6560000	cost: 1.94
Update 6570000	cost: 1.94
Update 6580000	cost: 1.96
Update 6590000	cost: 1.93
Update 6600000	cost: 1.94
Update 6610000	cost: 1.94
Update 6620000	cost: 1.94
Update 6630000	cost: 1.93
Update 6640000	cost: 1.93
Update 6650000	cost: 1.96
Update 6660000	cost: 1.93
Update 6670000	cost: 1.92
Update 6680000	cost: 1.95
Update 6690000	cost: 1.94
Update 6700000	cost: 1.93
Update 6710000	cost: 1.94
Update 6720000	cost: 1.96
Update 6730000	cost: 1.92
Update 6740000	cost: 1.93
Update 67500

The ``similar_words`` can be used to find related words of the ``word``.

In [32]:
def lookup_word_idx(word, word_dict):
    try:
        return np.argwhere(np.array(word_dict) == word)[0][0]
    except:
        raise Exception("No such word in dict: {}".format(word))

def similar_words(embeddings, word, word_dict, hits):
    word_idx = lookup_word_idx(word, word_dict)
    similarity_scores = embeddings @ embeddings[word_idx]
    similar_word_idxs = np.argsort(-similarity_scores)    
    return [word_dict[i] for i in similar_word_idxs[:hits]]

In [33]:
print('\n\nTraining cost: {0:>2.2f}\n\n'.format(J))

sample_words = ['knight', 'holy', 'grail']

Vp_norm = Vp / as_matrix(np.linalg.norm(Vp , axis=1))
for w in sample_words:
    similar = similar_words(Vp_norm, w, train_dict, 5)
    print('Words similar to {}: {}'.format(w, ", ".join(similar)))



Training cost: 1.93


Words similar to knight: knight, #, you, act, ll
Words similar to holy: holy, Black, saw, Beast, clunk
Words similar to grail: grail, separate, taunting, chops, shaped


#### References

[1] Jeffrey Pennington, Richard Socher, and Christopher D. Manning. 2014. GloVe: Global Vectors for Word Representation. 